<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a Random Forest model for my content refresh task.

Random Forest is suitable because content refresh decisions depend on several signals together, such as content age, days since the last update, impressions, average position, CTR, and word count.

A simple rule uses fixed thresholds, but Random Forest can learn more complex patterns from these signals. The goal is to rank pages that are more likely to need a content review.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-holdout split.

Pages from the same client should not appear in both the training and test data. This gives a more honest evaluation because the model is tested on clients it did not see during training.

This helps measure whether the model can generalize instead of simply remembering the training data.

In [2]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/shweta-1202/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())

Current folder: /content/flyrank-ml-internship


In [3]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will compare the Random Forest with my Week-4 baseline using the same Precision@50 metric.

Precision@50 tells me how many of the top 50 pages selected for review are actually declining.

A higher Precision@50 means the review list contains more useful pages.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# Create the target
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

# Features available before the decision
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y = df["is_declining"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Predictions
predicted_probability = model.predict_proba(X_test)[:, 1]

# Precision@50
top50_indices = np.argsort(
    -predicted_probability
)[:50]

model_precision_50 = y_test.iloc[
    top50_indices
].mean()

print("Random Forest Precision@50:",
      round(model_precision_50, 3))

Random Forest Precision@50: 0.92


In [6]:
# Baseline rule
stale = (
    df["days_since_last_update"] >= 180
).astype(int)

visible = (
    df["impressions_90d"] >= 500
).astype(int)

df["baseline_score"] = (
    stale * visible * df["impressions_90d"]
)

# Calculate baseline on the test rows
baseline_test = df.loc[
    X_test.index,
    "baseline_score"
]

baseline_top50 = np.argsort(
    -baseline_test.values
)[:50]

baseline_precision_50 = y_test.iloc[
    baseline_top50
].mean()

print("Baseline Precision@50:",
      round(baseline_precision_50, 3))

print("Random Forest Precision@50:",
      round(model_precision_50, 3))

Baseline Precision@50: 0.64
Random Forest Precision@50: 0.92


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model can make mistakes because a page may have signals that look like a declining page but may not actually need a refresh.

For example, an old page can still perform well, while a newer page can decline for other reasons.

The model mainly uses measurable signals such as impressions, content age, update age, average position, CTR, and word count. These signals help prioritize pages for human review, but they do not prove that a page needs to be updated.

In [7]:
# Show which features the Random Forest considers important

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance)

                  feature  importance
2         impressions_90d    0.267474
3            avg_position    0.250564
0        content_age_days    0.163440
5              word_count    0.162496
4                     ctr    0.113446
1  days_since_last_update    0.042580


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.